# repositorio-sincronizar.ipynb — traz o GitHub pro Drive

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

O código vive no **GitHub**; o Colab lê do **Drive**. Este notebook é a ponte:
clona o repositório e copia `modulos/`, `notebooks/` e `dados_lexico/` por cima
do que está no Drive.

Rode **sempre que o repositório mudar** — e desconfie se fizer semanas que você
não roda.

## Por que isso precisa existir

Em 29/ago o Drive estava **56 commits atrás**: faltavam 10 notebooks (o portão
de qualidade, a compilação, os dois da Bíblia, os quatro do chinês) e 7
módulos, e os que existiam eram de 19–23/ago. Um teste rodado assim executa
código velho e falha por motivo que não existe mais no repositório — o pior
tipo de depuração.

## Por que copiar por CIMA, e não apagar e recriar

`cp` por cima escreve **no mesmo arquivo** do Drive: o id não muda, e o link do
Colab que você tem salvo continua abrindo o notebook certo. Apagar e recriar
daria um arquivo novo, com id novo, e todos os seus links quebrariam de uma vez.

## O que este notebook NÃO toca

Só `pipeline/`. Nada de `videos/`, `assets/` ou planilha — mídia e estoque não
estão no git e não têm o que sincronizar. A direção é **uma só**: GitHub → Drive.
Se você editou um notebook direto no Colab e não levou pro git, essa edição é
sobrescrita — a célula 3 te mostra o que vai mudar antes de mudar.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP                                                         ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil, subprocess, hashlib, json
from pathlib import Path

REPO = "https://github.com/alanabdmorais/narrated_video"
CLONE = Path("/content/repo_narrated_video")

if CLONE.exists():
    shutil.rmtree(CLONE)
print(f"⬇️  clonando {REPO}")
subprocess.run(["git", "clone", "--depth", "1", "--quiet", REPO, str(CLONE)], check=True)

commit = subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%h %cs %s"],
                        capture_output=True, text=True).stdout.strip()
print(f"✅ {commit}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ a mesma dos outros notebooks

# O que sincronizar. Tudo sob pipeline/ — o resto (videos/, assets/, planilhas)
# não está no git e não tem o que sincronizar.
PASTAS = ["modulos", "notebooks", "dados_lexico"]

DESTINO = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline")
ORIGEM  = CLONE / "pipeline"

# ── Documentação: vem de assets/, vai pra pipeline/documentacao/ ──────────
# A documentação das cores (central de decisão, card de legenda, colinha do
# YouTube) é gerada do cores.py e mora em assets/ -- pasta que este notebook
# NÃO sincroniza, porque lá também estão o biblia-almeida.csv de 4,5 MB, a
# marca e as planilhas, que não têm por que ocupar o Drive.
#
# O efeito colateral só apareceu quando alguém foi procurar: os documentos
# existiam, estavam em dia com o cores.py, e não havia como abri-los pelo
# Colab. Documentação que ninguém alcança é documentação que não existe.
#
# A lista é EXPLÍCITA, não um glob, pra nada novo entrar sem alguém decidir.
# E o destino fica DENTRO de pipeline/, preservando a regra de que este
# notebook nunca escreve fora dela -- é o que garante que ele jamais toque
# no seu vídeo ou no seu áudio.
DOCUMENTACAO = (
    "central-decisao-cores.html",       # as 20 classes e as 21 cores
    "central-decisao-cores-zh.html",    # a mesma, com chinês
    "card-legenda-cores.html",          # as 2 telas que abrem/fecham o vídeo
    "card-legenda-cores-zh.html",
    "colinha-emojis-youtube.html",      # blocos pra descrição do vídeo
    "jornadas-artifact.html",           # o mapa dos notebooks
    "card_legenda_cores_1.png",         # o card em 1920x1080, que entra no vídeo
    "card_legenda_cores_2.png",
    "card_legenda_cores_zh_1.png",
    "card_legenda_cores_zh_2.png",
)
ORIGEM_DOCS  = CLONE / "assets"
DESTINO_DOCS = DESTINO / "documentacao"

# ── Quem mora só no Drive DE PROPÓSITO ─────────────────────────────────────
# A célula 3 lista os arquivos que existem no Drive e não no repositório. Sem
# essa lista ela chama todos de suspeitos ("ou é lixo, confira e apague à
# mão") -- e dois deles não são: um é a Bíblia inteira, o outro este notebook
# acabou de escrever. Aviso que grita em cima de arquivo legítimo ensina a
# ignorar o aviso, e aí o dia em que aparecer lixo de verdade ele passa batido.
GERADOS_NO_DRIVE = {
    "dados_lexico/web-biblia.json":
        "a Bíblia inteira, baixada pelo biblia-texto-baixar — grande, e regerável",
}

# ── Trava: o Drive montado é o do projeto? ─────────────────────────────────
# O Colab monta o Drive da conta com que VOCÊ entrou nele, que nem sempre é a
# conta do projeto. Numa conta sem a pasta `narrated_video`, um sync sem trava
# criaria a árvore do zero e copiaria os 62 arquivos pra lá: tudo apareceria
# como "novo", a conferência por hash passaria (comparando a cópia com a
# origem, não com o Drive) e você só descobriria depois, procurando arquivo
# que não chegou. Aconteceu em 29/ago -- por isso esta célula existe.
#
# A trava é simples: o destino tem que JÁ existir e JÁ ter arquivo dentro.
# Sincronizar é atualizar uma pasta que existe, nunca criar uma nova.
print("📂 O que a conta montada tem em MyDrive:")
raiz_montada = Path("/content/drive/MyDrive")
for item in sorted(raiz_montada.iterdir())[:15]:
    print(f"     {'📁' if item.is_dir() else '  '} {item.name}")

problema = None
if not DESTINO.exists():
    problema = f"{DESTINO} não existe"
else:
    vazias = [p for p in PASTAS[:2] if not (DESTINO / p).exists()
              or not any((DESTINO / p).iterdir())]
    if vazias:
        problema = f"{', '.join(vazias)} não existe(m) ou está(ão) vazia(s) em {DESTINO}"

if problema:
    raise SystemExit(
        f"\n❌ PAREI ANTES DE ESCREVER — {problema}.\n\n"
        f"   Quase sempre é CONTA ERRADA: o Colab montou o Drive de outra\n"
        f"   conta sua. Confira a lista acima — se não tem '{PASTA_DRIVE_RAIZ}',\n"
        f"   é isso. Troque de conta no Colab (canto superior direito),\n"
        f"   reinicie o ambiente e rode de novo.\n\n"
        f"   Este notebook ATUALIZA uma pasta existente; ele nunca cria a\n"
        f"   pasta do projeto do zero.")

achados = {p: len(list((DESTINO / p).glob("*"))) if (DESTINO / p).exists() else 0
           for p in PASTAS}
print(f"\n✅ Pasta do projeto encontrada: {DESTINO}")
for pasta, n in achados.items():
    print(f"     {pasta}/: {n} arquivo(s)")
print(f"\nde   {ORIGEM}")
print(f"para {DESTINO}")

# ── A ABA que você está olhando é a versão que está no Drive? ──────────────
# O Colab lê o .ipynb quando você ABRE a aba e guarda as células na memória.
# Ele não relê o arquivo a cada execução. Então, depois que uma rodada troca
# este notebook no Drive, a aba continua rodando o código de ontem -- por
# quantas vezes você mandar rodar.
#
# O aviso da célula 4 só dispara na rodada em que a cópia acontece. Quem roda
# DEPOIS não recebe aviso nenhum: o sync diz "0 arquivos, tudo idêntico", e
# está certo -- o Drive está mesmo em dia. Só quem está velho é a aba. Aí o
# _manifesto.txt nunca é escrito e ninguém entende por quê.
#
# A conferência não precisa de sentinela nem número de versão: o texto DESTA
# célula tem que ser IGUAL a alguma célula do .ipynb que está no Drive.
#
# Igual, e não "estar contido": a primeira versão desta checagem procurava o
# texto dentro do arquivo inteiro, e passava justamente no caso que ela existe
# pra pegar -- quando a versão nova só ACRESCENTA linhas no fim de uma célula,
# a versão velha é um prefixo dela e continua sendo substring do arquivo.
try:
    from IPython import get_ipython
    _minha_celula = get_ipython().user_ns["In"][-1].strip()
    _meu_arquivo = DESTINO / "notebooks" / "repositorio-sincronizar.ipynb"
    _no_drive = json.loads(_meu_arquivo.read_text(encoding="utf-8"))
    _celulas_do_drive = {"".join(c["source"]).strip() for c in _no_drive["cells"]}
    _confere = _minha_celula in _celulas_do_drive
except Exception as _e:          # sem IPython, arquivo ausente, JSON estranho
    _confere, _erro = None, _e

if _confere is False:
    print()
    print("═" * 62)
    print("⚠️  A ABA ESTÁ VELHA — o código que está rodando não é o que")
    print("    está no Drive. Quase sempre é isto: uma rodada anterior")
    print("    atualizou este notebook, e o Colab continua com a versão")
    print("    que carregou quando você abriu a aba.")
    print()
    print("    Feche a aba SEM SALVAR (salvar grava o velho por cima do")
    print("    novo), abra o notebook de novo pelo Drive e rode.")
    print()
    print("    (Se você editou alguma célula aqui e não salvou, é isso")
    print("     também — e salvar agora é o certo, não o errado.)")
    print("═" * 62)
elif _confere is None:
    print(f"\n   (não deu pra conferir se esta aba está atualizada: {_erro})")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 O QUE VAI MUDAR — confira ANTES de copiar                     ║
# ╚══════════════════════════════════════════════════════════════════╝
# Nada é escrito nesta célula. Ela existe pra você ver, antes, se algum
# arquivo que você editou direto no Colab está prestes a ser sobrescrito.

def _sha(caminho):
    return hashlib.sha256(caminho.read_bytes()).hexdigest()

plano = {"novo": [], "muda": [], "igual": [], "so_no_drive": []}

for pasta in PASTAS:
    org, dst = ORIGEM / pasta, DESTINO / pasta
    if not org.exists():
        print(f"⚠️  {pasta}/ não existe no repositório — pulando")
        continue
    no_repo = {p.name for p in org.iterdir() if p.is_file()}
    for nome in sorted(no_repo):
        a, b = org / nome, dst / nome
        if not b.exists():
            plano["novo"].append(f"{pasta}/{nome}")
        elif _sha(a) != _sha(b):
            plano["muda"].append(f"{pasta}/{nome}")
        else:
            plano["igual"].append(f"{pasta}/{nome}")
    if dst.exists():
        for p in sorted(dst.iterdir()):
            if p.is_file() and p.name not in no_repo and not p.name.startswith("."):
                plano["so_no_drive"].append(f"{pasta}/{p.name}")

# Documentação: mesma conferência, origem e destino diferentes.
plano_docs = {"novo": [], "muda": [], "igual": [], "sumido": []}
for nome in DOCUMENTACAO:
    a, b = ORIGEM_DOCS / nome, DESTINO_DOCS / nome
    if not a.exists():
        plano_docs["sumido"].append(nome)      # está na lista mas não no repo
    elif not b.exists():
        plano_docs["novo"].append(nome)
    elif _sha(a) != _sha(b):
        plano_docs["muda"].append(nome)
    else:
        plano_docs["igual"].append(nome)

print(f"➕ {len(plano['novo'])} novo(s) — não existem no Drive ainda")
for f in plano["novo"]:  print(f"     {f}")
print(f"\n♻️  {len(plano['muda'])} vai(vão) ser sobrescrito(s) pelo repositório")
for f in plano["muda"]:  print(f"     {f}")
print(f"\n✅ {len(plano['igual'])} já idêntico(s) — serão pulados")

print(f"\n📄 documentação → {DESTINO_DOCS.name}/: "
      f"{len(plano_docs['novo'])} nova(s), {len(plano_docs['muda'])} atualizada(s), "
      f"{len(plano_docs['igual'])} já igual(is)")
for f in plano_docs["novo"] + plano_docs["muda"]:
    print(f"     {f}")
if plano_docs["sumido"]:
    print(f"   ⚠️  {len(plano_docs['sumido'])} na lista DOCUMENTACAO mas não no repositório:")
    for f in plano_docs["sumido"]:
        print(f"     {f}   (renomeado? tire da lista ou corrija o nome)")
esperados = [f for f in plano["so_no_drive"] if f in GERADOS_NO_DRIVE]
suspeitos = [f for f in plano["so_no_drive"] if f not in GERADOS_NO_DRIVE]

print(f"\n📦 {len(esperados)} só no Drive, e é assim mesmo")
for f in esperados: print(f"     {f} — {GERADOS_NO_DRIVE[f]}")

print(f"\n👻 {len(suspeitos)} só no Drive e não sei o que é — NÃO serão apagados")
for f in suspeitos: print(f"     {f}")
if suspeitos:
    print("\n   (ou é lixo de uma versão antiga, ou é algo que nunca foi pro git.")
    print("    Este notebook não apaga nada — confira e apague à mão se for lixo.")
    print("    Se for legítimo, some com ele daqui adicionando em GERADOS_NO_DRIVE.)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 COPIAR — só rode depois de conferir a célula acima            ║
# ╚══════════════════════════════════════════════════════════════════╝

# Lido ANTES do laço: a cópia abaixo sobrescreve este próprio arquivo.
try:
    _CODIGO_ANTES = [ "".join(c["source"]).strip()
                      for c in json.loads((DESTINO / "notebooks" / "repositorio-sincronizar.ipynb")
                                          .read_text(encoding="utf-8"))["cells"] ]
except Exception:
    _CODIGO_ANTES = None

copiados = 0
for caminho in plano["novo"] + plano["muda"]:
    pasta, nome = caminho.split("/", 1)
    # `parents=False`: a pasta do projeto já foi conferida na Configuração.
    # Com parents=True, um destino errado seria criado em vez de acusado.
    (DESTINO / pasta).mkdir(exist_ok=True)
    # copyfile (e não copy2) escreve NO MESMO arquivo do Drive quando ele já
    # existe -- o id não muda, e o link do Colab que você tem salvo continua
    # valendo. Apagar e recriar quebraria todos os links de uma vez.
    shutil.copyfile(ORIGEM / pasta / nome, DESTINO / pasta / nome)
    copiados += 1
    print(f"   {caminho}")

# `parents=True` aqui é seguro: pipeline/ já foi validada na Configuração,
# e documentacao/ é subpasta dela. A trava de conta continua valendo.
DESTINO_DOCS.mkdir(parents=True, exist_ok=True)
for nome in plano_docs["novo"] + plano_docs["muda"]:
    shutil.copyfile(ORIGEM_DOCS / nome, DESTINO_DOCS / nome)
    copiados += 1
    print(f"   documentacao/{nome}")

print(f"\n✅ {copiados} arquivo(s) sincronizado(s) ({len(plano['igual'])} já estavam iguais)")
print(f"   Drive agora em: {commit}")

# ── Este notebook se atualizou? Então esta execução ainda é a versão VELHA ──
# O Colab carregou o código na memória quando você abriu o arquivo. Se a
# cópia acabou de trocar o arquivo no Drive, as células que estão rodando
# continuam sendo as antigas -- e qualquer coisa que a versão nova faça (como
# gravar o _manifesto.txt) simplesmente não acontece, sem erro nenhum.
# Comparar BYTES aqui dá alarme falso: um .ipynb difere por saída de célula e
# metadado do Colab mesmo com o código igual, e aí o aviso dispara toda vez --
# inclusive quando a aba já está em dia. Quem decide é o CÓDIGO das células,
# mesmo critério da conferência da Configuração. As duas discordaram uma vez;
# a precisa estava certa.
_EU = "repositorio-sincronizar.ipynb"

def _fontes(caminho):
    return [ "".join(c["source"]).strip()
             for c in json.loads(Path(caminho).read_text(encoding="utf-8"))["cells"] ]

_meu_codigo_mudou = False
if any(c.endswith(_EU) for c in plano["novo"] + plano["muda"]):
    try:
        _meu_codigo_mudou = _fontes(ORIGEM / "notebooks" / _EU) != _CODIGO_ANTES
    except Exception:
        _meu_codigo_mudou = True   # não deu pra comparar: avisa, que é o seguro

if _meu_codigo_mudou:
    print()
    print("═" * 62)
    print("⚠️  ESTE NOTEBOOK SE ATUALIZOU NESTA RODADA.")
    print("    O que acabou de rodar é a versão ANTIGA, que já estava")
    print("    carregada na memória. Feche a aba, abra o notebook de novo")
    print("    (Arquivo → Abrir → Google Drive) e rode outra vez.")
    print("    Só a segunda passada roda o código novo.")
    print("═" * 62)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ✅ CONFERIR — o Drive ficou byte a byte igual ao repositório?     ║
# ╚══════════════════════════════════════════════════════════════════╝
# Copiar pra um Drive montado pode falhar por cota, sessão expirada ou
# arquivo aberto -- e o shutil nem sempre grita. Aqui a prova é o hash.

divergentes = []
for pasta in PASTAS:
    org = ORIGEM / pasta
    if not org.exists():
        continue
    for a in sorted(p for p in org.iterdir() if p.is_file()):
        b = DESTINO / pasta / a.name
        if not b.exists() or _sha(a) != _sha(b):
            divergentes.append(f"{pasta}/{a.name}")

for nome in DOCUMENTACAO:
    a, b = ORIGEM_DOCS / nome, DESTINO_DOCS / nome
    if a.exists() and (not b.exists() or _sha(a) != _sha(b)):
        divergentes.append(f"documentacao/{nome}")

if divergentes:
    print(f"❌ {len(divergentes)} arquivo(s) NÃO bateram — rode a célula de copiar de novo:")
    for f in divergentes:
        print(f"     {f}")
else:
    print("✅ Drive idêntico ao repositório, byte a byte.")
    print(f"   {commit}")